# MACEField architecture: field conditioning and response derivatives

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mdi-group/mace-field-tutorial/blob/main/tutorials/macefield/01_macefield_architecture.ipynb)


This standalone notebook explains how the `MACEField` model in this checkout handles periodic systems in a homogeneous electric field. It follows the implementation in [`mace/modules/extensions.py`](../../mace/modules/extensions.py). Its numerical derivative check uses the BaTiO₃ reference-data model trained in notebook 02.

The derivative check can be skipped until the trained model exists; the architecture and implementation sections are useful on their own.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

PACKAGE_URL = "https://github.com/mdi-group/mace-field.git"
TUTORIAL_URL = "https://github.com/mdi-group/mace-field-tutorial.git"


def clone_checkout(path, url, branch):
    path = Path(path)
    if not (path / ".git").is_dir():
        if path.exists() and any(path.iterdir()):
            raise RuntimeError(f"Refusing to clone over a non-empty directory: {path}")
        subprocess.run(
            ["git", "clone", "--depth", "1", "--branch", branch, url, str(path)],
            check=True,
        )


if IN_COLAB:
    REPO = Path("/content/mace-field")
    TUTORIAL_REPO = Path("/content/mace-field-tutorial")
    clone_checkout(REPO, PACKAGE_URL, "clean")
    clone_checkout(TUTORIAL_REPO, TUTORIAL_URL, "main")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "--editable", str(REPO)],
        check=True,
    )
else:
    CWD = Path.cwd().resolve()
    candidates = (CWD, *CWD.parents)
    TUTORIAL_REPO = next(
        (path for path in candidates
         if (path / "tutorials/macefield/README.md").is_file()),
        None,
    )
    if TUTORIAL_REPO is None:
        raise RuntimeError(
            "Run from a mace-field-tutorial checkout, or open this notebook in Colab."
        )
    package_candidates = [
        Path(os.environ["MACEFIELD_REPO"]).expanduser().resolve()
        if os.environ.get("MACEFIELD_REPO") else None,
        TUTORIAL_REPO.parent / "mace-field",
        *candidates,
    ]
    REPO = next(
        (path.resolve() for path in package_candidates
         if path is not None and (path / "mace/modules/extensions.py").is_file()),
        None,
    )
    if REPO is None:
        raise RuntimeError(
            "Clone mdi-group/mace-field beside this tutorial checkout, or set MACEFIELD_REPO."
        )
    os.environ["MACEFIELD_REPO"] = str(REPO)

os.environ["MACEFIELD_REPO"] = str(REPO)
sys.path.insert(0, str(REPO))
sys.path.insert(0, str(TUTORIAL_REPO))
print("MACEField source:", REPO)
print("Tutorial assets:", TUTORIAL_REPO)
print("Runtime:", "Google Colab" if IN_COLAB else "local Jupyter")

from IPython.display import Image, display

TUTORIAL_DIR = TUTORIAL_REPO / "tutorials/macefield"
display(Image(filename=str(TUTORIAL_REPO / "tutorials/figures/architecture_macefield.png"), width=850))


## What you should know first

This lesson assumes familiarity with MACE message passing, equivariant features, and energy/force training. By the end you should be able to:

1. explain where the electric field enters the MACEField trunk;
2. derive polarization, Born effective charges, and dielectric response from the same energy;
3. trace the field and response values through the model and ASE calculator; and
4. name the symmetry, unit, and data checks needed before using a field-conditioned model.

## Start from the MACE energy model

For atomic positions $\mathbf R$, ordinary MACE builds local equivariant features through interaction and product layers, then sums atomic scalar readouts:

$$
E_\theta(\mathbf R)=\sum_i \epsilon_{\theta,i}(\mathbf h_i).
$$

The scalar energy is invariant to translations, rotations, and permutations of identical atoms. Its positional derivative gives conservative forces, $\mathbf F_i=-\partial E/\partial\mathbf r_i$. The hidden features $\mathbf h_i$ carry rotation representations (irreps), rather than being unrelated Cartesian channels.

## Condition the representation on a global field

A homogeneous electric field $\boldsymbol{\mathcal E}$ is a polar vector, so it transforms in the $1o$ irrep. MACEField broadcasts one 3-vector to the atoms in each graph. At each non-final interaction layer, a fully connected tensor product combines the current hidden features with this vector, maps the result back to the hidden irreps, and subtracts a learned linear correction from the node features.

In this checkout the relevant modules are named `field_feats` and `field_linear`. The model still predicts a scalar total energy; the field-conditioned features make that energy depend on both structure and field. This is not a fixed-charge $-\mathbf E\cdot\boldsymbol\mu$ add-on and does not, by itself, impose a long-range electrostatic model.

In [ ]:
display(Image(filename=str(TUTORIAL_REPO / "tutorials/figures/architecture_macefield.png"), width=850))


The diagram is a teaching schematic. The implementation is authoritative: `MACEField` subclasses `ScaleShiftMACE`, creates `field_feats`/`field_linear` for `num_interactions - 1` layers, and applies them inside the interaction loop. The final interaction is not field-updated by these modules.

## One energy generates the field response

Write the learned field-dependent energy as $E_\theta(\mathbf R,\boldsymbol{\mathcal E})$. For a periodic cell of volume $\Omega$, this implementation defines polarization by

$$
\mathbf P=-\frac{1}{\Omega}\frac{\partial E_\theta}{\partial\boldsymbol{\mathcal E}}.
$$

The same scalar energy gives forces, $\mathbf F_i=-\partial E_\theta/\partial\mathbf r_i$. Further derivatives give

$$
Z^*_{i,\alpha\beta}=\Omega\frac{\partial P_\alpha}{\partial r_{i\beta}},\qquad
\chi_{\alpha\beta}=\frac{1}{\epsilon_0}\frac{\partial P_\alpha}{\partial\mathcal E_\beta}.
$$

These relations tie energy, force, polarization, Born effective charges, and dielectric susceptibility together. The model can still be inaccurate; consistency of derivatives does not establish agreement with reference physics.

## Units and symmetry checks

| Quantity | Meaning | Units used by this workflow |
| --- | --- | --- |
| $\mathcal E$ | homogeneous electric field | V Å⁻¹ |
| $E$ | cell energy | eV |
| $\mathbf F$ | atomic force | eV Å⁻¹ |
| $\mathbf P$ | polarization | e Å⁻² |
| $Z^*$ | Born effective charge tensor | e |
| $\chi$ | dimensionless electric susceptibility | $\epsilon_0^{-1}\partial P/\partial E$ |

Rotate the crystal and field together: energy should remain unchanged, while polarization and forces should rotate with the structure. Reversing only the field is a different physical input. For periodic polarization, compare consistently chosen polarization branches.

## Follow the code path

In [`mace/modules/extensions.py`](../../mace/modules/extensions.py), inspect `MACEField.__init__` and `MACEField.forward`. Follow these operations in order:

1. read the explicit argument or `data["electric_field"]`, then reshape it to one vector per graph;
2. compute ordinary species, geometric, and equivariant features;
3. broadcast each graph's field to its atoms;
4. apply the field tensor product and linear correction at intermediate layers;
5. sum scalar readouts to get the cell energy; and
6. use autograd for forces and requested field-response derivatives.

The calculator path is in [`mace/calculators/mace.py`](../../mace/calculators/mace.py): `_resolve_electric_field` selects the field, and the results mapping exposes `polarization`, `becs`, and `polarizability`.

In [ ]:
import inspect
from mace.modules.extensions import MACEField

source = inspect.getsource(MACEField.__init__)
print(source)

The code below checks the sign and volume normalization of polarization against a finite difference of the trained model's energy, using a real BaTiO₃ structure and a small field perturbation around its labelled field. This checks implementation consistency, not model accuracy or field transfer.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ase.io import read
from mace.calculators import MACECalculator

DATA = TUTORIAL_DIR / "data"
MODEL = TUTORIAL_DIR / "runs/bto_macefield_training/MACEField-BaTiO3-reference_run-23.model"
if not MODEL.is_file():
    print("Run notebook 02 to train the reference-data model, then rerun this cell.")
else:
    sample = read(DATA / "macefield_batio3_reference_train.extxyz", index=0)
    field0 = np.asarray(sample.info["REF_electric_field"], dtype=float)
    direction = field0 / np.linalg.norm(field0)
    field_offsets = np.linspace(-2.0e-4, 2.0e-4, 9)
    energies, p_parallel = [], []
    calc = MACECalculator(
        model_paths=str(MODEL), model_type="MACEField", device="cpu",
        default_dtype="float64", electric_field=field0,
        compute_polarization=True, compute_becs=False,
        compute_polarizability=False,
    )
    for offset in field_offsets:
        calc.electric_field = field0 + offset * direction
        calc.reset()
        sample.calc = calc
        energies.append(sample.get_potential_energy())
        p_parallel.append(float(np.dot(calc.results["polarization"], direction)))

    energies = np.asarray(energies)
    p_parallel = np.asarray(p_parallel)
    p_parallel_fd = -np.gradient(energies, field_offsets) / sample.get_volume()
    fig, ax = plt.subplots(figsize=(6.5, 4))
    ax.plot(field_offsets, p_parallel, "o-", label="autograd polarization")
    ax.plot(field_offsets, p_parallel_fd, "s--", label="finite difference of energy")
    ax.set(xlabel="field perturbation along labelled [111] direction (V/Å)",
           ylabel="polarization projection (e/Å²)",
           title="Energy derivative check on a reference structure")
    ax.grid(alpha=0.25)
    ax.legend()
    plt.show()
    print("max derivative mismatch (e/Å²):",
          float(np.max(np.abs(p_parallel - p_parallel_fd))))

## Exercises

1. Change the field perturbation size and repeat the derivative check. What changes as finite-difference truncation and numerical precision compete?
2. Inspect `get_polarization` in [`mace/modules/utils.py`](../../mace/modules/utils.py). Why is the negative sign part of the definition?
3. Design a rotation-covariance check for forces and polarization. Rotate both the structure and field, and compare the rotated outputs.
4. Explain why correct finite-difference derivatives are necessary but insufficient evidence that a model predicts a ferroelectric energy landscape.

## Next notebooks

- [Train MACEField on the copied BaTiO₃ reference data](02_train_macefield_batio3.ipynb)
- [Run the exploratory finite-field MLMD cycle](03_finite_field_mlmd_hysteresis.ipynb)

The training set has one field vector and no switching-path data. The MD notebook shows the workflow while keeping that coverage limit explicit.